# 📘 Apostila de Estudo — APIs e Coleta de Dados

**Linguagem:** Python 3 · **Formato:** Jupyter Notebook · **Nível:** Iniciante → Intermediário

---

## Sumário

1. [APIs](#1)
2. [REST APIs](#2)
3. [Web Scraping](#3)
4. [Working with Files (Trabalhando com Arquivos)](#4)
5. [Work with Different File Formats (Diferentes Formatos de Arquivo)](#5)
6. [Exemplos de "como usar" e "aplicações"](#6)
7. [Questões Conceituais](#7)
8. [Exercícios com Gabarito](#8)

---

## Como usar esta apostila

- Leia a teoria em cada seção e **execute as células de código** (Shift + Enter), na ordem.
- Células que acessam a internet estão protegidas por `try/except`: se estiver offline, elas exibem uma mensagem em vez de falhar.
- Ao final, resolva as questões e os exercícios **antes** de consultar o gabarito.

### Instalação das bibliotecas necessárias

Execute a célula abaixo uma única vez (remova o `#` se precisar instalar).

In [ ]:
# !pip install requests beautifulsoup4 pandas openpyxl lxml pyyaml pyarrow matplotlib

import sys
print("Python:", sys.version.split()[0])
for lib in ["requests", "bs4", "pandas", "openpyxl", "lxml", "yaml", "pyarrow", "matplotlib"]:
    try:
        __import__(lib)
        print(f"✅ {lib} disponível")
    except ImportError:
        print(f"⚠️  {lib} não instalado")

<a id="1"></a>
# 1. APIs (Application Programming Interfaces)

## 1.1 O que é uma API?

Uma **API** (Interface de Programação de Aplicações) é um **contrato** que define como dois programas se comunicam. Ela expõe um conjunto de *funções, rotas ou métodos* que podem ser chamados por outro software, **sem que este precise conhecer a implementação interna**.

> **Analogia:** Em um restaurante, você (cliente) não entra na cozinha. Você faz o pedido ao garçom (API), que leva a solicitação à cozinha (servidor) e traz o prato (resposta). O cardápio é a **documentação** da API.

## 1.2 Por que APIs são importantes para Coleta de Dados?

| Vantagem | Descrição |
|---|---|
| **Dados estruturados** | Normalmente entregues em JSON ou XML, fáceis de processar. |
| **Confiabilidade** | Contrato estável e documentado (ao contrário do HTML de um site, que muda com frequência). |
| **Legalidade** | Uso autorizado e regulado pelos termos do provedor. |
| **Automação** | Permite coletas periódicas e integração entre sistemas. |

## 1.3 Tipos de APIs

- **APIs de biblioteca / locais** – ex.: `math.sqrt()`, `pandas.read_csv()` – chamadas dentro do mesmo programa.
- **APIs de Sistema Operacional** – ex.: módulo `os` do Python, que conversa com o SO.
- **APIs Web (remotas)** – acessadas via rede usando HTTP: **REST**, **GraphQL**, **SOAP**, **gRPC**, **WebSockets**.

## 1.4 O protocolo HTTP em 5 minutos

A comunicação Web segue o modelo **cliente-servidor** e usa o protocolo **HTTP/HTTPS**.

**Requisição (Request)** é composta por:
1. **Método** (verbo): `GET`, `POST`, `PUT`, `PATCH`, `DELETE`…
2. **URL** (endereço do recurso): `https://api.exemplo.com/usuarios/42?ativo=true`
3. **Cabeçalhos (Headers)**: metadados, ex.: `Content-Type`, `Authorization`, `User-Agent`
4. **Corpo (Body)**: dados enviados (geralmente em POST/PUT), ex.: JSON.

**Resposta (Response)** é composta por:
1. **Código de status** (ex.: `200`, `404`, `500`)
2. **Cabeçalhos**
3. **Corpo** (JSON, HTML, XML, imagem…)

### Anatomia de uma URL

```
https://api.exemplo.com:443/v1/produtos/15?categoria=livros&pagina=2#topo
└─┬──┘ └──────┬───────┘└┬┘└───────┬──────┘ └──────────┬───────────┘└─┬─┘
esquema      host     porta     caminho (path)   query string      fragmento
```

### Principais códigos de status HTTP

| Faixa | Significado | Exemplos |
|---|---|---|
| 2xx | Sucesso | `200 OK`, `201 Created`, `204 No Content` |
| 3xx | Redirecionamento | `301 Moved Permanently`, `304 Not Modified` |
| 4xx | Erro do cliente | `400 Bad Request`, `401 Unauthorized`, `403 Forbidden`, `404 Not Found`, `429 Too Many Requests` |
| 5xx | Erro do servidor | `500 Internal Server Error`, `503 Service Unavailable` |

## 1.5 JSON — a "língua" das APIs

**JSON** (JavaScript Object Notation) é um formato de texto leve para troca de dados. Mapeia diretamente para tipos Python:

| JSON | Python |
|---|---|
| `object {}` | `dict` |
| `array []` | `list` |
| `string` | `str` |
| `number` | `int` / `float` |
| `true` / `false` | `True` / `False` |
| `null` | `None` |

## 1.6 Autenticação em APIs

- **Sem autenticação** (APIs públicas).
- **API Key**: chave enviada em header (`X-API-Key`) ou query (`?apikey=...`).
- **Bearer Token / OAuth 2.0**: `Authorization: Bearer <token>`.
- **Basic Auth**: usuário e senha codificados em Base64.

> ⚠️ **Nunca** coloque chaves de API diretamente no código que será compartilhado. Use variáveis de ambiente (`os.environ`) ou arquivos `.env` fora do controle de versão.

In [ ]:
# Exemplo 1.1 — Primeira requisição com a biblioteca requests
import requests

url = "https://api.github.com"  # API pública do GitHub (raiz)

try:
    resposta = requests.get(url, timeout=10)
    print("Status code :", resposta.status_code)
    print("Content-Type:", resposta.headers.get("Content-Type"))
    print("Tipo do corpo:", type(resposta.json()))
    # Mostra apenas as 5 primeiras chaves do JSON
    for chave in list(resposta.json().keys())[:5]:
        print("  •", chave)
except requests.exceptions.RequestException as e:
    print("Sem conexão ou erro de rede:", type(e).__name__)

In [ ]:
# Exemplo 1.2 — JSON <-> Python (sem precisar de internet)
import json

texto_json = """
{
  "nome": "Ana",
  "idade": 28,
  "linguagens": ["Python", "SQL"],
  "ativo": true,
  "endereco": null
}
"""

dados = json.loads(texto_json)          # str  -> dict   (deserialização / parse)
print(type(dados), dados["linguagens"][0], dados["endereco"])

novo_texto = json.dumps(dados, indent=2, ensure_ascii=False)  # dict -> str (serialização)
print(novo_texto)

In [ ]:
# Exemplo 1.3 — Boas práticas: chaves de API via variável de ambiente
import os

api_key = os.environ.get("MINHA_API_KEY")  # retorna None se não existir

if api_key:
    print("Chave carregada (mascarada):", api_key[:3] + "..." + api_key[-2:])
else:
    print("Variável MINHA_API_KEY não definida. Defina com: export MINHA_API_KEY=xxxx")

# Como ela seria usada em um header:
headers = {"Authorization": f"Bearer {api_key or 'TOKEN_AQUI'}"}
print(headers.keys())

<a id="2"></a>
# 2. REST APIs

## 2.1 O que é REST?

**REST** (*Representational State Transfer*) é um **estilo arquitetural** proposto por Roy Fielding (2000) para sistemas distribuídos. Uma API que segue seus princípios é chamada **RESTful**.

### Os 6 princípios (constraints) do REST

1. **Cliente-Servidor**: separação de responsabilidades.
2. **Stateless (sem estado)**: cada requisição contém toda a informação necessária; o servidor não guarda "sessão".
3. **Cacheável**: respostas podem indicar se podem ser armazenadas em cache.
4. **Interface uniforme**: recursos identificados por URIs, manipulados por representações (JSON), mensagens autodescritivas.
5. **Sistema em camadas**: intermediários (proxies, gateways) são transparentes.
6. **Código sob demanda** (opcional).

## 2.2 Recursos e Verbos HTTP (CRUD)

Em REST, tudo é um **recurso** (substantivo), identificado por uma URL. As **ações** são expressas pelos verbos HTTP:

| Operação | Verbo HTTP | Exemplo de endpoint | Idempotente? |
|---|---|---|---|
| **C**reate | `POST` | `POST /posts` | Não |
| **R**ead | `GET` | `GET /posts` · `GET /posts/1` | Sim |
| **U**pdate (total) | `PUT` | `PUT /posts/1` | Sim |
| **U**pdate (parcial) | `PATCH` | `PATCH /posts/1` | Não necessariamente |
| **D**elete | `DELETE` | `DELETE /posts/1` | Sim |

> **Idempotente**: executar a mesma operação várias vezes produz o mesmo resultado que executá-la uma vez.

### Boas práticas de design de endpoints

- Substantivos no **plural**: `/usuarios`, não `/getUsuario`.
- Hierarquia: `/usuarios/42/pedidos/7`.
- Filtro, ordenação e paginação via **query string**: `/produtos?categoria=livros&ordem=preco&pagina=2&limite=50`.
- Versionamento: `/v1/...`, `/v2/...`.

## 2.3 Parâmetros: Path vs Query vs Body vs Header

| Tipo | Onde vai | Uso típico | Exemplo |
|---|---|---|---|
| **Path** | Na URL, parte do caminho | Identificar recurso | `/usuarios/42` |
| **Query** | Após `?` | Filtrar, paginar, ordenar | `?pagina=2&limite=10` |
| **Body** | Corpo da requisição | Enviar dados (POST/PUT) | `{"nome": "Ana"}` |
| **Header** | Cabeçalhos | Autenticação, formato | `Authorization`, `Accept` |

## 2.4 Paginação e Rate Limiting

- **Paginação**: APIs não devolvem tudo de uma vez. Estratégias comuns: `?page=2&per_page=100` (offset) ou **cursor** (`?after=abc123`).
- **Rate limiting**: limite de requisições por período. Ao exceder, recebe-se `429 Too Many Requests`. Verifique headers como `X-RateLimit-Remaining` e `Retry-After`. Use `time.sleep()` e **backoff exponencial**.

## 2.5 REST × GraphQL × SOAP (visão rápida)

| | REST | GraphQL | SOAP |
|---|---|---|---|
| Formato | JSON (geralmente) | JSON | XML |
| Endpoints | Vários (por recurso) | Um único (`/graphql`) | Um único |
| Flexibilidade da consulta | Fixa por endpoint | Cliente escolhe campos | Fixa (WSDL) |
| Complexidade | Baixa | Média | Alta |

## 2.6 A biblioteca `requests`

```python
requests.get(url, params={}, headers={}, timeout=10)
requests.post(url, json={}, data={}, headers={})
requests.put(url, json={})
requests.patch(url, json={})
requests.delete(url)
```

Atributos importantes do objeto `Response`:
- `.status_code`, `.ok` (True se 2xx/3xx), `.headers`, `.url`
- `.text` (string), `.content` (bytes), `.json()` (dict/list)
- `.raise_for_status()` → lança exceção se status for 4xx/5xx

Nos exemplos usaremos a **JSONPlaceholder** (`https://jsonplaceholder.typicode.com`), uma API REST pública e gratuita para testes.

In [ ]:
# Exemplo 2.1 — GET: listar e buscar um recurso
import requests

BASE = "https://jsonplaceholder.typicode.com"

try:
    # GET coleção
    r = requests.get(f"{BASE}/posts", timeout=10)
    r.raise_for_status()
    posts = r.json()
    print(f"Total de posts: {len(posts)}")
    print("Primeiro post:", {k: posts[0][k] for k in ("userId", "id", "title")})

    # GET recurso específico (path parameter)
    r = requests.get(f"{BASE}/posts/1", timeout=10)
    print("\nPost 1 →", r.json()["title"])

    # GET com query parameters (filtro)
    r = requests.get(f"{BASE}/comments", params={"postId": 1}, timeout=10)
    print("\nURL montada pelo requests:", r.url)
    print("Comentários do post 1:", len(r.json()))
except requests.exceptions.RequestException as e:
    print("Erro de rede:", type(e).__name__)

In [ ]:
# Exemplo 2.2 — POST, PUT, PATCH e DELETE
import requests

BASE = "https://jsonplaceholder.typicode.com"
novo_post = {"title": "Aprendendo REST", "body": "Coleta de dados com Python", "userId": 7}

try:
    r = requests.post(f"{BASE}/posts", json=novo_post, timeout=10)   # json= serializa e define Content-Type
    print("POST   →", r.status_code, r.json())

    r = requests.put(f"{BASE}/posts/1", json={**novo_post, "id": 1}, timeout=10)
    print("PUT    →", r.status_code, r.json()["title"])

    r = requests.patch(f"{BASE}/posts/1", json={"title": "Só o título mudou"}, timeout=10)
    print("PATCH  →", r.status_code, r.json()["title"])

    r = requests.delete(f"{BASE}/posts/1", timeout=10)
    print("DELETE →", r.status_code)
except requests.exceptions.RequestException as e:
    print("Erro de rede:", type(e).__name__)

In [ ]:
# Exemplo 2.3 — Tratamento robusto de erros e retries com backoff exponencial
import requests, time

def get_com_retry(url, tentativas=3, espera_inicial=1, **kwargs):
    """Faz GET com novas tentativas em caso de erro de rede, 5xx ou 429."""
    for i in range(tentativas):
        try:
            r = requests.get(url, timeout=10, **kwargs)
            if r.status_code == 429:
                espera = int(r.headers.get("Retry-After", espera_inicial * 2**i))
                print(f"Rate limit! Aguardando {espera}s...")
                time.sleep(espera); continue
            if 500 <= r.status_code < 600:
                raise requests.exceptions.HTTPError(f"Erro servidor {r.status_code}")
            r.raise_for_status()
            return r
        except requests.exceptions.RequestException as e:
            espera = espera_inicial * 2**i
            print(f"Tentativa {i+1} falhou ({type(e).__name__}). Aguardando {espera}s...")
            time.sleep(espera)
    raise RuntimeError("Todas as tentativas falharam.")

try:
    r = get_com_retry("https://jsonplaceholder.typicode.com/users/1")
    print("Sucesso:", r.json()["name"])
    r = get_com_retry("https://jsonplaceholder.typicode.com/rota-inexistente", tentativas=1)
except Exception as e:
    print("Resultado final:", type(e).__name__, "-", str(e)[:80])

In [ ]:
# Exemplo 2.4 — Paginação: percorrendo todas as páginas de uma API
import requests

def coletar_paginado(url, per_page=20, max_paginas=5):
    todos = []
    for pagina in range(1, max_paginas + 1):
        r = requests.get(url, params={"_page": pagina, "_limit": per_page}, timeout=10)
        r.raise_for_status()
        lote = r.json()
        if not lote:              # página vazia → acabou
            break
        todos.extend(lote)
        print(f"Página {pagina}: {len(lote)} itens")
    return todos

try:
    dados = coletar_paginado("https://jsonplaceholder.typicode.com/posts")
    print("Total coletado:", len(dados))
except requests.exceptions.RequestException as e:
    print("Erro de rede:", type(e).__name__)

In [ ]:
# Exemplo 2.5 — Sessões e headers (User-Agent, Accept) + convertendo em DataFrame
import requests, pandas as pd

sessao = requests.Session()                # reutiliza conexão TCP e mantém headers/cookies
sessao.headers.update({
    "User-Agent": "ApostilaAPIs/1.0 (estudo)",
    "Accept": "application/json",
})

try:
    usuarios = sessao.get("https://jsonplaceholder.typicode.com/users", timeout=10).json()
    df = pd.json_normalize(usuarios)       # "achata" JSON aninhado (address.city, company.name...)
    print(df[["id", "name", "email", "address.city", "company.name"]].head())
except requests.exceptions.RequestException as e:
    print("Erro de rede:", type(e).__name__)

## 2.7 Construindo sua própria REST API (visão geral com Flask)

Consumir APIs é metade da história; saber **como uma API é construída** ajuda a entender seu comportamento. Abaixo, uma mini API com Flask (não execute dentro do notebook — salve como `app.py` e rode `python app.py`).

```python
from flask import Flask, jsonify, request, abort

app = Flask(__name__)
livros = {1: {"id": 1, "titulo": "Dom Casmurro", "autor": "Machado de Assis"}}

@app.get("/livros")                      # GET coleção
def listar():
    return jsonify(list(livros.values()))

@app.get("/livros/<int:id>")             # GET recurso (path param)
def obter(id):
    return jsonify(livros.get(id)) if id in livros else abort(404)

@app.post("/livros")                     # POST → 201 Created
def criar():
    dados = request.get_json()
    novo_id = max(livros) + 1
    livros[novo_id] = {"id": novo_id, **dados}
    return jsonify(livros[novo_id]), 201

@app.delete("/livros/<int:id>")          # DELETE → 204 No Content
def remover(id):
    livros.pop(id, None)
    return "", 204

if __name__ == "__main__":
    app.run(debug=True)                  # http://127.0.0.1:5000
```

Teste com: `curl http://127.0.0.1:5000/livros` ou `requests.get("http://127.0.0.1:5000/livros")`.

<a id="3"></a>
# 3. Web Scraping

## 3.1 O que é?

**Web Scraping** é a extração automatizada de dados de páginas web, lendo o **HTML** que o navegador renderiza. É usado quando **não existe API** disponível para os dados desejados.

| | API | Web Scraping |
|---|---|---|
| Fonte | Endpoint estruturado (JSON) | HTML da página |
| Estabilidade | Alta | Baixa (layout muda → código quebra) |
| Autorização | Explícita (termos de uso) | Depende do `robots.txt` e dos Termos |
| Esforço | Baixo | Médio/Alto |

## 3.2 Ética e Legalidade — leia antes de raspar!

1. **Verifique o `robots.txt`** (`https://site.com/robots.txt`) — indica o que os robôs podem ou não acessar.
2. **Leia os Termos de Uso** do site.
3. **Não sobrecarregue o servidor**: use `time.sleep()` entre requisições.
4. **Identifique-se** com um `User-Agent` honesto.
5. **Respeite dados pessoais** (LGPD/GDPR).
6. Prefira **APIs oficiais** sempre que existirem.

## 3.3 Revisão de HTML

```html
<html>
  <head><title>Minha Página</title></head>
  <body>
    <div id="conteudo" class="principal">
      <h1>Título</h1>
      <p class="texto">Parágrafo <a href="/link">com link</a></p>
      <ul>
        <li>Item 1</li>
        <li>Item 2</li>
      </ul>
      <table>
        <tr><th>Nome</th><th>Preço</th></tr>
        <tr><td>Caneta</td><td>2,50</td></tr>
      </table>
    </div>
  </body>
</html>
```

- **Tags**: `<div>`, `<p>`, `<a>`, `<table>`…
- **Atributos**: `id` (único), `class` (reutilizável), `href`, `src`.
- **Árvore DOM**: elementos aninhados (pai → filhos → irmãos).

## 3.4 Ferramentas em Python

| Ferramenta | Função |
|---|---|
| `requests` | Baixar o HTML |
| `BeautifulSoup` (bs4) | Parsear e navegar no HTML |
| `lxml` | Parser rápido; suporta XPath |
| `pandas.read_html()` | Extrair tabelas HTML direto para DataFrame |
| `Selenium` / `Playwright` | Sites dinâmicos (JavaScript) — controlam um navegador |
| `Scrapy` | Framework completo para crawlers em larga escala |

### Métodos essenciais do BeautifulSoup

```python
soup.find("tag", class_="x")        # primeiro elemento
soup.find_all("tag", attrs={...})   # lista de elementos
soup.select("div.principal p a")    # seletores CSS (lista)
soup.select_one("#conteudo h1")     # seletor CSS (primeiro)
el.text / el.get_text(strip=True)   # texto
el["href"] / el.get("href")         # atributo
el.parent, el.find_next_sibling()   # navegação
```

### Seletores CSS mais usados

| Seletor | Significado |
|---|---|
| `p` | todas as tags `<p>` |
| `.texto` | classe `texto` |
| `#conteudo` | id `conteudo` |
| `div > p` | `<p>` filho direto de `<div>` |
| `a[href^="http"]` | links cujo href começa com http |
| `tr:nth-child(2)` | segunda linha |

In [ ]:
# Exemplo 3.1 — Parseando HTML local com BeautifulSoup (funciona offline)
from bs4 import BeautifulSoup

html = """
<html><head><title>Loja Exemplo</title></head>
<body>
  <div id="produtos">
    <div class="produto" data-id="101">
      <h2 class="nome">Notebook</h2><span class="preco">R$ 3.500,00</span>
      <a href="/produto/101">Ver</a>
    </div>
    <div class="produto" data-id="102">
      <h2 class="nome">Mouse</h2><span class="preco">R$ 80,00</span>
      <a href="/produto/102">Ver</a>
    </div>
    <div class="produto esgotado" data-id="103">
      <h2 class="nome">Teclado</h2><span class="preco">R$ 150,00</span>
      <a href="/produto/103">Ver</a>
    </div>
  </div>
</body></html>
"""

soup = BeautifulSoup(html, "html.parser")

print("Título da página:", soup.title.text)
print("Primeiro h2:", soup.find("h2").text)
print("Qtde de produtos:", len(soup.find_all("div", class_="produto")))

print("\nLista de produtos:")
for prod in soup.select("div.produto"):
    nome  = prod.select_one(".nome").get_text(strip=True)
    preco = prod.select_one(".preco").get_text(strip=True)
    link  = prod.find("a")["href"]
    esgotado = "esgotado" in prod.get("class", [])
    print(f"  [{prod['data-id']}] {nome:10s} {preco:14s} {link}  {'(esgotado)' if esgotado else ''}")

In [ ]:
# Exemplo 3.2 — Limpando e convertendo dados raspados para DataFrame
import re, pandas as pd

def preco_para_float(texto):
    """'R$ 3.500,00' -> 3500.0"""
    numero = re.sub(r"[^\d,]", "", texto).replace(",", ".")
    return float(numero)

registros = []
for prod in soup.select("div.produto"):
    registros.append({
        "id": int(prod["data-id"]),
        "nome": prod.select_one(".nome").get_text(strip=True),
        "preco": preco_para_float(prod.select_one(".preco").text),
        "disponivel": "esgotado" not in prod.get("class", []),
    })

df_produtos = pd.DataFrame(registros)
print(df_produtos)
print("\nPreço médio dos disponíveis:", df_produtos[df_produtos.disponivel].preco.mean())

In [ ]:
# Exemplo 3.3 — Extraindo tabelas HTML direto com pandas.read_html
import pandas as pd
from io import StringIO

html_tabela = """
<table id="ranking">
  <thead><tr><th>Posição</th><th>Linguagem</th><th>Popularidade (%)</th></tr></thead>
  <tbody>
    <tr><td>1</td><td>Python</td><td>28.1</td></tr>
    <tr><td>2</td><td>Java</td><td>15.7</td></tr>
    <tr><td>3</td><td>JavaScript</td><td>9.0</td></tr>
  </tbody>
</table>
"""
tabelas = pd.read_html(StringIO(html_tabela))   # retorna LISTA de DataFrames (uma por <table>)
print(f"{len(tabelas)} tabela(s) encontrada(s)")
print(tabelas[0])

In [ ]:
# Exemplo 3.4 — Scraping real: verificando robots.txt e raspando uma página pública
import requests, time
from bs4 import BeautifulSoup
from urllib.robotparser import RobotFileParser

URL = "https://quotes.toscrape.com/"        # site criado justamente para praticar scraping
HEADERS = {"User-Agent": "ApostilaAPIs/1.0 (estudo educacional)"}

try:
    # 1) robots.txt
    rp = RobotFileParser(); rp.set_url(URL + "robots.txt"); rp.read()
    print("Permitido raspar '/'?", rp.can_fetch(HEADERS["User-Agent"], URL))

    # 2) baixar e parsear
    r = requests.get(URL, headers=HEADERS, timeout=10); r.raise_for_status()
    soup_q = BeautifulSoup(r.text, "html.parser")

    citacoes = []
    for q in soup_q.select("div.quote"):
        citacoes.append({
            "texto": q.select_one("span.text").get_text(strip=True),
            "autor": q.select_one("small.author").get_text(strip=True),
            "tags": [t.text for t in q.select("a.tag")],
        })
    print(f"\n{len(citacoes)} citações na página 1. Exemplo:")
    print(citacoes[0])

    # 3) seguir paginação (educadamente, com pausa)
    proximo = soup_q.select_one("li.next a")
    print("\nLink da próxima página:", proximo["href"] if proximo else None)
    time.sleep(1)   # boas maneiras
except Exception as e:
    print("Erro de rede:", type(e).__name__)

## 3.5 Sites dinâmicos (JavaScript)

Muitos sites carregam dados via JavaScript **depois** que o HTML inicial chega. Nesses casos, `requests` recebe um HTML "vazio". Soluções:

1. **Inspecionar a aba Network** do navegador (F12) — muitas vezes o site consome uma **API interna** em JSON, que você pode chamar diretamente com `requests`. ✅ Melhor opção.
2. **Selenium / Playwright** — automatizam um navegador real que executa o JS.

```python
# Exemplo com Selenium (requer navegador + driver instalados)
from selenium import webdriver
from selenium.webdriver.common.by import By

driver = webdriver.Chrome()
driver.get("https://exemplo.com")
elementos = driver.find_elements(By.CSS_SELECTOR, "div.item")
print([e.text for e in elementos])
driver.quit()
```

<a id="4"></a>
# 4. Working with Files (Trabalhando com Arquivos)

Depois de coletar dados (via API ou scraping), precisamos **persistir** e **ler** esses dados. Aqui entra a manipulação de arquivos.

## 4.1 A função `open()`

```python
arquivo = open("caminho/arquivo.txt", mode="r", encoding="utf-8")
```

| Modo | Significado |
|---|---|
| `"r"` | Leitura (padrão). Erro se não existir. |
| `"w"` | Escrita. **Apaga** o conteúdo se existir; cria se não existir. |
| `"a"` | Append — acrescenta ao final. |
| `"x"` | Criação exclusiva — erro se já existir. |
| `"b"` | Binário (`"rb"`, `"wb"`) — imagens, PDFs, Parquet… |
| `"t"` | Texto (padrão). |
| `"+"` | Leitura e escrita (`"r+"`, `"w+"`). |

## 4.2 O gerenciador de contexto `with`

Sempre use `with`: ele **fecha o arquivo automaticamente**, mesmo se ocorrer um erro.

```python
with open("dados.txt", "w", encoding="utf-8") as f:
    f.write("linha 1\n")
# aqui o arquivo já está fechado
```

## 4.3 Métodos de leitura e escrita

| Método | Descrição |
|---|---|
| `f.read()` | Lê todo o conteúdo como string |
| `f.read(n)` | Lê `n` caracteres |
| `f.readline()` | Lê uma linha |
| `f.readlines()` | Lista com todas as linhas |
| `for linha in f:` | Itera linha a linha (eficiente para arquivos grandes) |
| `f.write(s)` | Escreve string |
| `f.writelines(lista)` | Escreve lista de strings (sem adicionar `\n`) |
| `f.seek(0)` / `f.tell()` | Move / consulta posição do cursor |

## 4.4 Encoding

Texto é armazenado como bytes; o **encoding** define a tradução. Use sempre **UTF-8**. Arquivos antigos do Windows/Excel em português costumam estar em `latin-1` (`ISO-8859-1`) ou `cp1252`. Sintoma de encoding errado: `Ã§`, `Ã£` no lugar de `ç`, `ã`.

## 4.5 Caminhos: `os` e `pathlib`

`pathlib.Path` (moderno, orientado a objetos) é preferível ao `os.path`.

| Tarefa | `pathlib` | `os` / `os.path` |
|---|---|---|
| Juntar caminhos | `Path("pasta") / "arq.txt"` | `os.path.join("pasta", "arq.txt")` |
| Existe? | `p.exists()` | `os.path.exists(p)` |
| É arquivo/pasta? | `p.is_file()` / `p.is_dir()` | `os.path.isfile()` / `isdir()` |
| Criar pasta | `p.mkdir(parents=True, exist_ok=True)` | `os.makedirs(p, exist_ok=True)` |
| Listar | `p.iterdir()`, `p.glob("*.csv")` | `os.listdir(p)`, `glob.glob()` |
| Nome / extensão | `p.name`, `p.stem`, `p.suffix` | `os.path.basename()`, `splitext()` |
| Apagar | `p.unlink()` | `os.remove()` |
| Tamanho | `p.stat().st_size` | `os.path.getsize()` |

In [ ]:
# Exemplo 4.1 — Escrever, acrescentar e ler um arquivo de texto
from pathlib import Path

pasta = Path("dados_apostila"); pasta.mkdir(exist_ok=True)
arq = pasta / "log.txt"

with open(arq, "w", encoding="utf-8") as f:          # 'w' cria/sobrescreve
    f.write("Linha 1: início da coleta\n")
    f.write("Linha 2: 100 registros obtidos\n")

with open(arq, "a", encoding="utf-8") as f:          # 'a' acrescenta
    f.write("Linha 3: coleta finalizada ✔\n")

with open(arq, "r", encoding="utf-8") as f:          # 'r' lê
    conteudo = f.read()
print(conteudo)

print("Tamanho:", arq.stat().st_size, "bytes | Existe?", arq.exists(), "| Extensão:", arq.suffix)

In [ ]:
# Exemplo 4.2 — Lendo linha a linha (eficiente para arquivos grandes) e contando
with open(arq, encoding="utf-8") as f:
    for numero, linha in enumerate(f, start=1):
        print(f"{numero:02d} | {linha.rstrip()} | {len(linha.split())} palavras")

# readlines() vs iteração
with open(arq, encoding="utf-8") as f:
    linhas = f.readlines()
print("\nTotal de linhas:", len(linhas))

In [ ]:
# Exemplo 4.3 — Encoding: o que acontece quando erramos
texto = "Coração, ação, informação"

with open(pasta / "utf8.txt", "w", encoding="utf-8") as f:   f.write(texto)
with open(pasta / "latin1.txt", "w", encoding="latin-1") as f: f.write(texto)

print("Bytes UTF-8   :", (pasta / "utf8.txt").stat().st_size)
print("Bytes Latin-1 :", (pasta / "latin1.txt").stat().st_size)

# Lendo o arquivo latin-1 como se fosse utf-8 → erro
try:
    print(open(pasta / "latin1.txt", encoding="utf-8").read())
except UnicodeDecodeError as e:
    print("UnicodeDecodeError:", str(e)[:60])

# Lendo utf-8 como latin-1 → mojibake (caracteres estranhos)
print("Mojibake:", open(pasta / "utf8.txt", encoding="latin-1").read())
print("Correto :", open(pasta / "latin1.txt", encoding="latin-1").read())

In [ ]:
# Exemplo 4.4 — Arquivos binários e navegação em diretórios com pathlib
import os

# binário: gravando bytes
with open(pasta / "dados.bin", "wb") as f:
    f.write(bytes([0x50, 0x59, 0x54, 0x48, 0x4F, 0x4E]))   # "PYTHON" em ASCII

with open(pasta / "dados.bin", "rb") as f:
    b = f.read()
print("Bytes lidos:", b, "→ decodificado:", b.decode("ascii"))

# listando a pasta
print("\nConteúdo da pasta:")
for p in sorted(pasta.iterdir()):
    print(f"  {p.name:12s} {p.stat().st_size:5d} bytes  {'[dir]' if p.is_dir() else ''}")

print("\nApenas .txt:", [p.name for p in pasta.glob("*.txt")])
print("Diretório atual:", os.getcwd())

<a id="5"></a>
# 5. Work with Different File Formats (Diferentes Formatos de Arquivo)

## 5.1 Visão geral

| Formato | Tipo | Estrutura | Melhor para | Bibliotecas |
|---|---|---|---|---|
| **TXT** | Texto | Livre | Logs, texto bruto | `open` |
| **CSV** | Texto | Tabular (linhas/colunas) | Planilhas simples, intercâmbio | `csv`, `pandas` |
| **JSON** | Texto | Hierárquica (dict/list) | APIs, configs, dados aninhados | `json`, `pandas` |
| **XML** | Texto | Hierárquica (tags) | Sistemas legados, SOAP, RSS | `xml.etree`, `lxml`, `bs4` |
| **YAML** | Texto | Hierárquica | Arquivos de configuração | `pyyaml` |
| **Excel (XLSX)** | Binário | Tabular, múltiplas abas | Relatórios corporativos | `pandas` + `openpyxl` |
| **Parquet** | Binário colunar | Tabular | Big Data, análise eficiente | `pandas` + `pyarrow` |
| **Pickle** | Binário | Objetos Python | Cache de objetos (⚠️ inseguro com fontes desconhecidas) | `pickle` |
| **SQLite** | Binário | Relacional | Banco de dados local | `sqlite3`, `pandas` |

## 5.2 CSV (Comma-Separated Values)

- Cada linha = um registro; campos separados por delimitador (`,` ou `;` no Brasil, por causa da vírgula decimal).
- Primeira linha geralmente é o **cabeçalho**.
- Campos com delimitador ou quebra de linha vão entre aspas: `"São Paulo, SP"`.
- Sem tipos: tudo é texto → conversão necessária.

## 5.3 JSON

- Aninhamento ilimitado; ideal para respostas de API.
- `json.load(f)` / `json.dump(obj, f)` → arquivos. `json.loads(s)` / `json.dumps(obj)` → strings.
- **JSON Lines** (`.jsonl`): um objeto JSON por linha — ótimo para streams e arquivos grandes.

## 5.4 XML

- Baseado em tags como HTML, mas com tags definidas pelo usuário.
- Elementos, atributos, texto. Ex.: `<livro id="1"><titulo>...</titulo></livro>`.
- Navegação com `ElementTree`: `root.findall("livro")`, `el.find("titulo").text`, `el.get("id")`.

## 5.5 Excel, Parquet e outros com pandas

`pandas` unifica a leitura/escrita: `pd.read_csv`, `pd.read_json`, `pd.read_excel`, `pd.read_parquet`, `pd.read_xml`, `pd.read_sql` e os respectivos `df.to_*()`.

In [ ]:
# Exemplo 5.1 — CSV com o módulo csv (escrita e leitura, com DictWriter/DictReader)
import csv

registros = [
    {"cidade": "São Paulo", "uf": "SP", "populacao": 12300000, "obs": "Maior cidade, capital"},
    {"cidade": "Rio de Janeiro", "uf": "RJ", "populacao": 6700000, "obs": ""},
    {"cidade": "Curitiba", "uf": "PR", "populacao": 1960000, "obs": "Frio; \"cidade modelo\""},
]
arq_csv = pasta / "cidades.csv"

with open(arq_csv, "w", newline="", encoding="utf-8") as f:   # newline="" evita linhas em branco no Windows
    writer = csv.DictWriter(f, fieldnames=registros[0].keys(), delimiter=";")
    writer.writeheader()
    writer.writerows(registros)

print(open(arq_csv, encoding="utf-8").read())

with open(arq_csv, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f, delimiter=";")
    for linha in reader:
        print(linha["cidade"], "→", int(linha["populacao"]) / 1e6, "milhões")   # note a conversão de tipo

In [ ]:
# Exemplo 5.2 — CSV com pandas (o jeito prático)
import pandas as pd

df = pd.read_csv(arq_csv, sep=";", encoding="utf-8")
print(df.dtypes, "\n")           # pandas infere tipos automaticamente
print(df)

df["pop_milhoes"] = df["populacao"] / 1e6
df.to_csv(pasta / "cidades_out.csv", index=False, sep=",", decimal=".")
print("\nSalvo:", (pasta / "cidades_out.csv").name)

# Parâmetros úteis de read_csv: sep, encoding, decimal, thousands, usecols, dtype, parse_dates, nrows, skiprows, na_values

In [ ]:
# Exemplo 5.3 — JSON: arquivo, JSON aninhado e JSON Lines
import json, pandas as pd

dados = {
    "coleta": "2026-09-16",
    "fonte": "api.exemplo.com",
    "itens": [
        {"id": 1, "nome": "Ana",   "contato": {"email": "ana@x.com", "tel": None}},
        {"id": 2, "nome": "Bruno", "contato": {"email": "bruno@x.com", "tel": "11-9999"}},
    ],
}

# gravar / ler arquivo JSON
with open(pasta / "coleta.json", "w", encoding="utf-8") as f:
    json.dump(dados, f, indent=2, ensure_ascii=False)
with open(pasta / "coleta.json", encoding="utf-8") as f:
    lido = json.load(f)
print("Fonte:", lido["fonte"], "| itens:", len(lido["itens"]))

# JSON aninhado → tabela
df = pd.json_normalize(lido["itens"])
print(df, "\n")

# JSON Lines: um objeto por linha
with open(pasta / "coleta.jsonl", "w", encoding="utf-8") as f:
    for item in dados["itens"]:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")
print(open(pasta / "coleta.jsonl", encoding="utf-8").read())
print(pd.read_json(pasta / "coleta.jsonl", lines=True))

In [ ]:
# Exemplo 5.4 — XML com xml.etree.ElementTree
import xml.etree.ElementTree as ET

xml_texto = """<?xml version="1.0" encoding="UTF-8"?>
<biblioteca>
  <livro id="1" ano="1899">
    <titulo>Dom Casmurro</titulo>
    <autor>Machado de Assis</autor>
    <preco moeda="BRL">39.90</preco>
  </livro>
  <livro id="2" ano="1956">
    <titulo>Grande Sertão: Veredas</titulo>
    <autor>João Guimarães Rosa</autor>
    <preco moeda="BRL">59.90</preco>
  </livro>
</biblioteca>"""

(pasta / "biblioteca.xml").write_text(xml_texto, encoding="utf-8")

tree = ET.parse(pasta / "biblioteca.xml")
root = tree.getroot()
print("Raiz:", root.tag, "| filhos:", len(root))

for livro in root.findall("livro"):
    print(f"[{livro.get('id')}] {livro.find('titulo').text} ({livro.get('ano')}) - "
          f"{livro.find('autor').text} - {float(livro.find('preco').text):.2f} {livro.find('preco').get('moeda')}")

# criando XML programaticamente
novo = ET.SubElement(root, "livro", id="3", ano="1928")
ET.SubElement(novo, "titulo").text = "Macunaíma"
ET.SubElement(novo, "autor").text = "Mário de Andrade"
tree.write(pasta / "biblioteca_v2.xml", encoding="utf-8", xml_declaration=True)
print("\nLivros após inserção:", len(root.findall("livro")))

# Alternativa direta com pandas (requer lxml):
import pandas as pd
try:
    print(pd.read_xml(pasta / "biblioteca.xml"))
except ImportError:
    print("pd.read_xml requer lxml")

In [ ]:
# Exemplo 5.5 — Excel (múltiplas abas) com pandas + openpyxl
import pandas as pd

df_cidades = pd.read_csv(arq_csv, sep=";")
df_resumo = df_cidades.groupby("uf", as_index=False)["populacao"].sum()

arq_xlsx = pasta / "relatorio.xlsx"
try:
    with pd.ExcelWriter(arq_xlsx, engine="openpyxl") as writer:
        df_cidades.to_excel(writer, sheet_name="Cidades", index=False)
        df_resumo.to_excel(writer, sheet_name="Resumo por UF", index=False)

    abas = pd.read_excel(arq_xlsx, sheet_name=None)      # None → dict com todas as abas
    print("Abas:", list(abas.keys()))
    print(abas["Resumo por UF"])
except ImportError:
    print("Instale openpyxl: pip install openpyxl")

In [ ]:
# Exemplo 5.6 — Parquet, Pickle, YAML e SQLite
import pandas as pd, pickle, sqlite3

# Parquet: colunar, comprimido, preserva tipos → ideal para grandes volumes
try:
    df_cidades.to_parquet(pasta / "cidades.parquet", index=False)
    print("Parquet:", pd.read_parquet(pasta / "cidades.parquet").shape,
          "|", (pasta / "cidades.parquet").stat().st_size, "bytes")
except ImportError:
    print("Parquet requer pyarrow: pip install pyarrow")

# Pickle: serializa QUALQUER objeto Python (⚠️ nunca carregue pickle de fonte não confiável)
obj = {"modelo": "v1", "parametros": [0.1, 0.2], "df": df_cidades}
with open(pasta / "cache.pkl", "wb") as f: pickle.dump(obj, f)
with open(pasta / "cache.pkl", "rb") as f: print("Pickle:", pickle.load(f)["parametros"])

# YAML: configurações legíveis
try:
    import yaml
    config = {"api": {"base_url": "https://api.exemplo.com", "timeout": 10, "retries": 3},
              "saida": {"formato": "csv", "pasta": "dados"}}
    with open(pasta / "config.yaml", "w") as f: yaml.safe_dump(config, f, sort_keys=False)
    print("YAML:\n" + open(pasta / "config.yaml").read())
    print("Lido:", yaml.safe_load(open(pasta / "config.yaml"))["api"]["timeout"])
except ImportError:
    print("YAML requer pyyaml: pip install pyyaml")

# SQLite: banco relacional em um único arquivo
con = sqlite3.connect(pasta / "dados.db")
df_cidades.to_sql("cidades", con, if_exists="replace", index=False)
print("\nSQLite:", pd.read_sql("SELECT uf, SUM(populacao) AS pop FROM cidades GROUP BY uf", con))
con.close()

## 5.6 Como escolher o formato?

```
Precisa ser lido por humanos / editado no Excel?  → CSV ou XLSX
Dados hierárquicos / resposta de API?             → JSON
Configuração de projeto?                          → YAML (ou JSON)
Grande volume + análise em pandas/Spark?          → Parquet
Integração com sistema legado / RSS / SOAP?       → XML
Consultas com filtros e relacionamentos?          → SQLite / banco SQL
Cache rápido de objetos Python (uso interno)?     → Pickle
```

<a id="6"></a>
# 6. Exemplos de "como usar" e "aplicações"

Nesta seção integramos tudo: **coletar → limpar → armazenar → analisar**.

## 6.1 Aplicações reais de APIs e coleta de dados

| Área | Aplicação | Fonte típica |
|---|---|---|
| Finanças | Cotações de moedas e ações, análise de risco | AwesomeAPI, Banco Central (SGS), Yahoo Finance |
| Dados públicos | Indicadores socioeconômicos, censo | IBGE, dados.gov.br, World Bank, Portal da Transparência |
| Clima | Previsão, agricultura, logística | Open-Meteo, OpenWeatherMap, INMET |
| Geolocalização | CEP → endereço, rotas, mapas | ViaCEP, Nominatim (OSM), Google Maps |
| Marketing | Monitoramento de preços e concorrência | Web scraping de e-commerces |
| Ciência de Dados | Datasets para ML, NLP | Kaggle API, Hugging Face, arXiv |
| DevOps | Monitoramento, integração contínua | GitHub API, Slack API |
| Saúde | Dados epidemiológicos | OpenDataSUS, WHO |

## 6.2 Pipeline típico de coleta

```
┌──────────┐   ┌──────────┐   ┌───────────┐   ┌────────────┐   ┌──────────┐
│ Extração │ → │ Validação│ → │ Transform.│ → │ Armazenam. │ → │ Análise  │
│ API/HTML │   │ status,  │   │ limpeza,  │   │ CSV/JSON/  │   │ pandas,  │
│          │   │ schema   │   │ tipos     │   │ Parquet/DB │   │ gráficos │
└──────────┘   └──────────┘   └───────────┘   └────────────┘   └──────────┘
```

In [ ]:
# Exemplo 6.1 — Consulta de CEP (ViaCEP) — API pública brasileira sem autenticação
import requests

def consultar_cep(cep):
    cep = "".join(filter(str.isdigit, cep))
    if len(cep) != 8:
        raise ValueError("CEP deve ter 8 dígitos")
    r = requests.get(f"https://viacep.com.br/ws/{cep}/json/", timeout=10)
    r.raise_for_status()
    dados = r.json()
    return None if dados.get("erro") else dados

try:
    for cep in ["01001-000", "70040-010", "99999-999"]:
        info = consultar_cep(cep)
        print(cep, "→", f"{info['logradouro']}, {info['bairro']}, {info['localidade']}/{info['uf']}" if info else "CEP não encontrado")
except requests.exceptions.RequestException as e:
    print("Erro de rede:", type(e).__name__)

In [ ]:
# Exemplo 6.2 — Cotações de moedas (AwesomeAPI) → DataFrame → CSV
import requests, pandas as pd
from datetime import datetime

try:
    r = requests.get("https://economia.awesomeapi.com.br/last/USD-BRL,EUR-BRL,BTC-BRL", timeout=10)
    r.raise_for_status()
    cot = r.json()                    # dict: {"USDBRL": {...}, "EURBRL": {...}, ...}

    df_cot = pd.DataFrame(cot.values())[["code", "codein", "name", "bid", "ask", "pctChange", "create_date"]]
    df_cot[["bid", "ask", "pctChange"]] = df_cot[["bid", "ask", "pctChange"]].astype(float)
    df_cot["coletado_em"] = datetime.now().isoformat(timespec="seconds")
    print(df_cot)

    # Append em um histórico CSV (cria cabeçalho só na primeira vez)
    hist = pasta / "cotacoes_historico.csv"
    df_cot.to_csv(hist, mode="a", index=False, header=not hist.exists())
    print("\nHistórico agora tem", sum(1 for _ in open(hist)) - 1, "linhas")
except requests.exceptions.RequestException as e:
    print("Erro de rede:", type(e).__name__)

In [ ]:
# Exemplo 6.3 — Dados públicos do IBGE: estados e regiões → JSON
import requests, pandas as pd

try:
    r = requests.get("https://servicodados.ibge.gov.br/api/v1/localidades/estados", timeout=15)
    r.raise_for_status()
    estados = pd.DataFrame(r.json())[["id", "sigla", "nome"]]
    estados["regiao"] = [e["regiao"]["nome"] for e in r.json()]
    print(estados.head())
    print("\nEstados por região:")
    print(estados.groupby("regiao")["sigla"].apply(list))
    estados.to_json(pasta / "estados_ibge.json", orient="records", force_ascii=False, indent=2)
except requests.exceptions.RequestException as e:
    print("Erro de rede:", type(e).__name__)

In [ ]:
# Exemplo 6.4 — Previsão do tempo (Open-Meteo, sem chave) e visualização
import requests, pandas as pd

try:
    params = {"latitude": -23.55, "longitude": -46.63,      # São Paulo
              "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
              "timezone": "America/Sao_Paulo", "forecast_days": 7}
    r = requests.get("https://api.open-meteo.com/v1/forecast", params=params, timeout=10)
    r.raise_for_status()
    d = r.json()["daily"]
    clima = pd.DataFrame(d).rename(columns={"time": "data", "temperature_2m_max": "t_max",
                                            "temperature_2m_min": "t_min", "precipitation_sum": "chuva_mm"})
    print(clima)

    import matplotlib.pyplot as plt
    ax = clima.plot(x="data", y=["t_max", "t_min"], marker="o", title="Previsão 7 dias — São Paulo")
    ax.set_ylabel("°C"); plt.xticks(rotation=45); plt.tight_layout(); plt.show()
except requests.exceptions.RequestException as e:
    print("Erro de rede:", type(e).__name__)
except ImportError:
    print("matplotlib não instalado")

In [ ]:
# Exemplo 6.5 — Projeto integrador: pipeline completo (API → validação → transformação → Parquet/CSV → análise)
import requests, pandas as pd, json, time
from pathlib import Path

def extrair(base="https://jsonplaceholder.typicode.com"):
    s = requests.Session(); s.headers["User-Agent"] = "ApostilaAPIs/1.0"
    users = s.get(f"{base}/users", timeout=10).json()
    posts = s.get(f"{base}/posts", timeout=10).json()
    todos = s.get(f"{base}/todos", timeout=10).json()
    return users, posts, todos

def validar(users, posts, todos):
    assert all({"id", "name", "email"} <= u.keys() for u in users), "schema de users inválido"
    assert all({"userId", "id", "title"} <= p.keys() for p in posts), "schema de posts inválido"
    print(f"✔ Validação OK: {len(users)} users, {len(posts)} posts, {len(todos)} todos")

def transformar(users, posts, todos):
    du = pd.json_normalize(users)[["id", "name", "email", "address.city", "company.name"]]
    du.columns = ["user_id", "nome", "email", "cidade", "empresa"]
    dp = pd.DataFrame(posts).groupby("userId").size().rename("n_posts")
    dt = pd.DataFrame(todos).groupby("userId")["completed"].mean().rename("pct_tarefas_concluidas")
    return du.merge(dp, left_on="user_id", right_index=True).merge(dt, left_on="user_id", right_index=True)

def carregar(df, pasta):
    df.to_csv(pasta / "usuarios_consolidado.csv", index=False)
    try: df.to_parquet(pasta / "usuarios_consolidado.parquet", index=False)
    except ImportError: pass
    with open(pasta / "metadata.json", "w") as f:
        json.dump({"linhas": len(df), "colunas": list(df.columns), "gerado_em": time.strftime("%Y-%m-%d %H:%M:%S")}, f, indent=2)

try:
    u, p, t = extrair()
    validar(u, p, t)
    df_final = transformar(u, p, t)
    carregar(df_final, pasta)
    print("\nTop 5 usuários por % de tarefas concluídas:")
    print(df_final.sort_values("pct_tarefas_concluidas", ascending=False).head()[["nome", "cidade", "n_posts", "pct_tarefas_concluidas"]])
    print("\nArquivos gerados:", sorted(x.name for x in pasta.glob("usuarios_consolidado*")))
except requests.exceptions.RequestException as e:
    print("Erro de rede:", type(e).__name__)

## 6.3 Checklist do bom coletor de dados

- [ ] Existe API oficial? Prefira-a ao scraping.
- [ ] Li a documentação: autenticação, limites, paginação, formato.
- [ ] Uso `timeout` em toda requisição.
- [ ] Trato erros (`try/except`, `raise_for_status`) e faço *retry* com *backoff*.
- [ ] Respeito `robots.txt`, termos de uso e uso `time.sleep()` entre chamadas.
- [ ] Chaves de API ficam em variáveis de ambiente, nunca no código.
- [ ] Salvo os dados **brutos** (raw) antes de transformar — permite reprocessar.
- [ ] Registro metadados: data da coleta, fonte, versão.
- [ ] Escolho o formato adequado (CSV/JSON/Parquet) ao volume e uso.

<a id="7"></a>
# 7. Questões Conceituais

Responda antes de consultar o gabarito no final da seção.

**Q1.** Defina API com suas palavras e explique a analogia do "garçom".

**Q2.** Qual a diferença entre os métodos HTTP `PUT` e `PATCH`? Qual deles é idempotente?

**Q3.** O que significa dizer que uma API REST é *stateless*? Que implicação prática isso tem para o cliente?

**Q4.** Um cliente recebeu os códigos `401`, `403`, `404` e `429` em requisições diferentes. Explique o significado de cada um e o que o cliente deve fazer em cada caso.

**Q5.** Diferencie *path parameter*, *query parameter*, *header* e *body*. Dê um exemplo de uso adequado de cada um.

**Q6.** Por que web scraping é considerado mais frágil do que consumir uma API? Cite dois cuidados éticos/legais indispensáveis.

**Q7.** Explique a diferença entre `soup.find()`, `soup.find_all()` e `soup.select()` no BeautifulSoup.

**Q8.** Por que devemos usar `with open(...)` em vez de `f = open(...)` seguido de `f.close()`?

**Q9.** Um arquivo CSV exportado do Excel em português apresenta `Ã§Ã£o` ao ser lido. Qual a provável causa e como corrigir?

**Q10.** Compare CSV, JSON e Parquet quanto a: legibilidade humana, suporte a dados hierárquicos e eficiência para grandes volumes.

**Q11.** O que é *rate limiting* e como um cliente bem comportado deve lidar com ele?

**Q12.** Explique o que faz `pd.json_normalize()` e em que situação ele é indispensável.

**Q13.** Por que `pickle.load()` de um arquivo de origem desconhecida é um risco de segurança?

**Q14.** Qual a diferença entre `json.load()` e `json.loads()`? E entre `json.dump()` e `json.dumps()`?

**Q15.** Um site carrega sua tabela de preços via JavaScript após o carregamento da página. `requests.get()` retorna um HTML sem os dados. Quais são as duas estratégias possíveis e qual é preferível?

## ✅ Gabarito — Questões Conceituais

**Q1.** API é um contrato/interface que define como programas se comunicam, expondo funcionalidades sem revelar a implementação interna. Na analogia, o cliente (aplicação) faz o pedido ao garçom (API), que o leva à cozinha (servidor) e retorna o prato (resposta); o cardápio é a documentação.

**Q2.** `PUT` substitui o recurso **inteiro** pela representação enviada (campos omitidos são apagados/zerados); `PATCH` aplica uma **modificação parcial**. `PUT` é idempotente (repetir produz o mesmo estado); `PATCH` não é necessariamente.

**Q3.** O servidor não armazena estado de sessão entre requisições; cada requisição deve conter tudo o que é necessário (token de autenticação, parâmetros, etc.). Na prática, o cliente precisa reenviar credenciais/contexto a cada chamada, e o servidor pode ser escalado horizontalmente com facilidade.

**Q4.** `401 Unauthorized`: não autenticado → enviar/renovar credenciais. `403 Forbidden`: autenticado, mas sem permissão → verificar escopo/permissões, não adianta repetir. `404 Not Found`: recurso/URL inexistente → conferir o endpoint ou ID. `429 Too Many Requests`: limite de taxa excedido → aguardar (`Retry-After`) e aplicar backoff.

**Q5.** *Path*: identifica o recurso (`/usuarios/42`). *Query*: filtra/pagina/ordena (`?pagina=2&uf=SP`). *Header*: metadados como autenticação e formato (`Authorization: Bearer x`, `Accept: application/json`). *Body*: dados a criar/atualizar (`POST /usuarios` com `{"nome": "Ana"}`).

**Q6.** O HTML é feito para exibição, não para consumo por máquinas; qualquer mudança de layout (classes, estrutura) quebra o scraper, enquanto APIs têm contrato versionado. Cuidados: respeitar `robots.txt` e Termos de Uso; não sobrecarregar o servidor (pausas entre requisições); respeitar dados pessoais (LGPD); identificar-se via `User-Agent`.

**Q7.** `find()` retorna o **primeiro** elemento que casa com tag/atributos (ou `None`); `find_all()` retorna **lista** de todos; `select()` usa **seletores CSS** (ex.: `div.produto > a[href]`) e retorna lista (`select_one()` retorna o primeiro).

**Q8.** O `with` garante que `close()` seja chamado automaticamente **mesmo se ocorrer uma exceção**, evitando vazamento de recursos, arquivos corrompidos (buffers não gravados) e bloqueios de arquivo. Também torna o código mais curto e legível.

**Q9.** O arquivo foi salvo em `latin-1`/`cp1252` (padrão do Excel no Windows em pt-BR) e lido como UTF-8 (ou vice-versa) — o chamado *mojibake*. Correção: `pd.read_csv(arq, encoding="latin-1")` ou `open(arq, encoding="cp1252")`. Alternativamente, reexportar o arquivo em UTF-8.

**Q10.** CSV: muito legível, apenas tabular (sem hierarquia), texto sem tipos, ineficiente para grandes volumes. JSON: legível, excelente para hierarquia/aninhamento, verboso e lento para grandes volumes tabulares. Parquet: binário (não legível), tabular (com suporte a tipos complexos), colunar e comprimido → o mais eficiente em espaço e velocidade para análise.

**Q11.** É o limite de requisições por período imposto pelo servidor. O cliente deve monitorar headers (`X-RateLimit-Remaining`, `Retry-After`), inserir pausas (`time.sleep`), aplicar *backoff exponencial* ao receber `429`, usar cache e paginar com tamanhos adequados.

**Q12.** Converte JSON aninhado (dicts dentro de dicts/listas) em um DataFrame "achatado", criando colunas como `address.city`. É indispensável quando a resposta da API possui objetos aninhados que precisam virar colunas tabulares.

**Q13.** Pickle pode conter instruções para reconstruir objetos arbitrários, incluindo chamadas a funções; um arquivo malicioso pode executar código arbitrário no momento da desserialização. Use pickle apenas para dados gerados por você mesmo; para intercâmbio, prefira JSON/Parquet.

**Q14.** `json.load(f)` lê de um **arquivo** (file object); `json.loads(s)` lê de uma **string** (o "s" é de *string*). Analogamente, `json.dump(obj, f)` escreve em arquivo e `json.dumps(obj)` retorna uma string.

**Q15.** (1) Inspecionar a aba *Network* do navegador para descobrir a API interna (XHR/fetch) que fornece os dados em JSON e chamá-la diretamente com `requests` — **preferível** (mais rápido, estável e leve). (2) Usar um navegador automatizado (Selenium/Playwright) que executa o JavaScript e depois raspar o DOM renderizado — mais lento e pesado.

<a id="8"></a>
# 8. Exercícios com Gabarito

Tente resolver cada exercício na célula "Sua resposta" antes de ver a solução.

---

### Exercício 1 — JSON básico
Dada a string JSON abaixo, converta para Python e imprima o nome de cada aluno com sua média (arredondada a 1 casa).

```python
alunos_json = '[{"nome":"Ana","notas":[8,9,7]},{"nome":"Bia","notas":[6,7.5,8]},{"nome":"Caio","notas":[10,9,9.5]}]'
```

In [ ]:
# Sua resposta — Exercício 1
alunos_json = '[{"nome":"Ana","notas":[8,9,7]},{"nome":"Bia","notas":[6,7.5,8]},{"nome":"Caio","notas":[10,9,9.5]}]'

In [ ]:
# ✅ Gabarito — Exercício 1
import json
alunos_json = '[{"nome":"Ana","notas":[8,9,7]},{"nome":"Bia","notas":[6,7.5,8]},{"nome":"Caio","notas":[10,9,9.5]}]'
for a in json.loads(alunos_json):
    print(f"{a['nome']}: {sum(a['notas'])/len(a['notas']):.1f}")

### Exercício 2 — GET com query parameters
Usando a JSONPlaceholder, obtenha todos os *todos* do usuário `userId=3` e informe: total de tarefas, quantas estão concluídas e o percentual de conclusão.

In [ ]:
# Sua resposta — Exercício 2

In [ ]:
# ✅ Gabarito — Exercício 2
import requests
try:
    r = requests.get("https://jsonplaceholder.typicode.com/todos", params={"userId": 3}, timeout=10)
    r.raise_for_status()
    todos = r.json()
    concluidas = sum(t["completed"] for t in todos)
    print(f"Total: {len(todos)} | Concluídas: {concluidas} | {concluidas/len(todos):.0%}")
except requests.exceptions.RequestException as e:
    print("Erro de rede:", type(e).__name__)

### Exercício 3 — Tratamento de status
Escreva uma função `buscar_post(id)` que retorne o dicionário do post ou `None` se receber `404`, e lance a exceção para qualquer outro erro. Teste com `id=5` e `id=9999`.

In [ ]:
# Sua resposta — Exercício 3

In [ ]:
# ✅ Gabarito — Exercício 3
import requests

def buscar_post(id):
    r = requests.get(f"https://jsonplaceholder.typicode.com/posts/{id}", timeout=10)
    if r.status_code == 404:
        return None
    r.raise_for_status()
    return r.json()

try:
    print(buscar_post(5)["title"])
    print(buscar_post(9999))
except requests.exceptions.RequestException as e:
    print("Erro de rede:", type(e).__name__)

### Exercício 4 — POST
Crie um novo *post* na JSONPlaceholder com título `"Exercício 4"`, corpo `"Enviado via requests"` e `userId=1`. Verifique se o status retornado é `201` e imprima o `id` gerado.

In [ ]:
# Sua resposta — Exercício 4

In [ ]:
# ✅ Gabarito — Exercício 4
import requests
try:
    r = requests.post("https://jsonplaceholder.typicode.com/posts",
                      json={"title": "Exercício 4", "body": "Enviado via requests", "userId": 1}, timeout=10)
    print("Status 201?", r.status_code == 201, "| id gerado:", r.json()["id"])
except requests.exceptions.RequestException as e:
    print("Erro de rede:", type(e).__name__)

### Exercício 5 — Web Scraping com HTML local
A partir do HTML abaixo, extraia uma lista de dicionários com `titulo`, `autor`, `ano` (int) e `link`, e mostre apenas os livros publicados **antes de 1950**.

```html
<ul class="livros">
  <li><a href="/l/1">Dom Casmurro</a> — <span class="autor">Machado de Assis</span> (<span class="ano">1899</span>)</li>
  <li><a href="/l/2">Vidas Secas</a> — <span class="autor">Graciliano Ramos</span> (<span class="ano">1938</span>)</li>
  <li><a href="/l/3">Grande Sertão: Veredas</a> — <span class="autor">Guimarães Rosa</span> (<span class="ano">1956</span>)</li>
  <li><a href="/l/4">A Hora da Estrela</a> — <span class="autor">Clarice Lispector</span> (<span class="ano">1977</span>)</li>
</ul>
```

In [ ]:
# Sua resposta — Exercício 5
html_ex5 = """
<ul class="livros">
  <li><a href="/l/1">Dom Casmurro</a> — <span class="autor">Machado de Assis</span> (<span class="ano">1899</span>)</li>
  <li><a href="/l/2">Vidas Secas</a> — <span class="autor">Graciliano Ramos</span> (<span class="ano">1938</span>)</li>
  <li><a href="/l/3">Grande Sertão: Veredas</a> — <span class="autor">Guimarães Rosa</span> (<span class="ano">1956</span>)</li>
  <li><a href="/l/4">A Hora da Estrela</a> — <span class="autor">Clarice Lispector</span> (<span class="ano">1977</span>)</li>
</ul>
"""

In [ ]:
# ✅ Gabarito — Exercício 5
from bs4 import BeautifulSoup
soup5 = BeautifulSoup(html_ex5, "html.parser")
livros = [{
    "titulo": li.a.get_text(strip=True),
    "autor": li.select_one(".autor").text,
    "ano": int(li.select_one(".ano").text),
    "link": li.a["href"],
} for li in soup5.select("ul.livros li")]

for l in livros:
    if l["ano"] < 1950:
        print(l)

### Exercício 6 — Tabela HTML → DataFrame → CSV
Use `pandas.read_html` para ler a tabela abaixo, adicione uma coluna `densidade` (habitantes/km²) e salve como `capitais.csv` com separador `;`. Depois releia o arquivo e mostre a capital mais densa.

```html
<table>
<tr><th>Capital</th><th>Populacao</th><th>Area_km2</th></tr>
<tr><td>São Paulo</td><td>12300000</td><td>1521</td></tr>
<tr><td>Rio de Janeiro</td><td>6700000</td><td>1200</td></tr>
<tr><td>Brasília</td><td>3100000</td><td>5760</td></tr>
<tr><td>Fortaleza</td><td>2700000</td><td>313</td></tr>
</table>
```

In [ ]:
# Sua resposta — Exercício 6
html_ex6 = """
<table>
<tr><th>Capital</th><th>Populacao</th><th>Area_km2</th></tr>
<tr><td>São Paulo</td><td>12300000</td><td>1521</td></tr>
<tr><td>Rio de Janeiro</td><td>6700000</td><td>1200</td></tr>
<tr><td>Brasília</td><td>3100000</td><td>5760</td></tr>
<tr><td>Fortaleza</td><td>2700000</td><td>313</td></tr>
</table>
"""

In [ ]:
# ✅ Gabarito — Exercício 6
import pandas as pd
from io import StringIO
from pathlib import Path

df6 = pd.read_html(StringIO(html_ex6))[0]
df6["densidade"] = (df6["Populacao"] / df6["Area_km2"]).round(1)
Path("dados_apostila").mkdir(exist_ok=True)
df6.to_csv("dados_apostila/capitais.csv", sep=";", index=False)

relido = pd.read_csv("dados_apostila/capitais.csv", sep=";")
mais_densa = relido.loc[relido["densidade"].idxmax()]
print(relido, "\n\nMais densa:", mais_densa["Capital"], f"({mais_densa['densidade']} hab/km²)")

### Exercício 7 — Arquivos de texto
Crie um arquivo `frases.txt` com 5 frases (uma por linha). Depois, leia o arquivo e escreva um novo arquivo `frases_maiusculas.txt` contendo apenas as frases com mais de 20 caracteres, em MAIÚSCULAS e numeradas. Imprima quantas frases foram gravadas.

In [ ]:
# Sua resposta — Exercício 7

In [ ]:
# ✅ Gabarito — Exercício 7
from pathlib import Path
pasta7 = Path("dados_apostila"); pasta7.mkdir(exist_ok=True)

frases = ["Python é divertido.", "APIs conectam sistemas diferentes entre si.", "Oi!",
          "Web scraping exige responsabilidade e ética.", "JSON é o formato padrão das APIs modernas."]
(pasta7 / "frases.txt").write_text("\n".join(frases), encoding="utf-8")

n = 0
with open(pasta7 / "frases.txt", encoding="utf-8") as fin, open(pasta7 / "frases_maiusculas.txt", "w", encoding="utf-8") as fout:
    for linha in fin:
        linha = linha.strip()
        if len(linha) > 20:
            n += 1
            fout.write(f"{n}. {linha.upper()}\n")
print(f"{n} frases gravadas\n")
print((pasta7 / "frases_maiusculas.txt").read_text(encoding="utf-8"))

### Exercício 8 — Conversão entre formatos
Leia o arquivo `capitais.csv` (Exercício 6) e gere, a partir dele: (a) `capitais.json` no formato de lista de registros; (b) `capitais.xlsx`; (c) `capitais.xml`. Confirme que os três arquivos existem e mostre seus tamanhos em bytes.

In [ ]:
# Sua resposta — Exercício 8

In [ ]:
# ✅ Gabarito — Exercício 8
import pandas as pd
from pathlib import Path
p = Path("dados_apostila")
df8 = pd.read_csv(p / "capitais.csv", sep=";")

df8.to_json(p / "capitais.json", orient="records", force_ascii=False, indent=2)
try:
    df8.to_excel(p / "capitais.xlsx", index=False)
except ImportError:
    print("openpyxl não instalado — pulando xlsx")
try:
    df8.to_xml(p / "capitais.xml", index=False, root_name="capitais", row_name="capital")
except ImportError:
    print("lxml não instalado — pulando xml")

for nome in ["capitais.json", "capitais.xlsx", "capitais.xml"]:
    arq = p / nome
    print(f"{nome:15s} existe={arq.exists()!s:5s} {arq.stat().st_size if arq.exists() else 0:6d} bytes")

### Exercício 9 — Paginação e consolidação
A API `https://jsonplaceholder.typicode.com/comments` aceita `_page` e `_limit`. Colete **todas** as páginas com `_limit=100` até receber uma página vazia, consolide em um DataFrame e responda: quantos comentários existem e qual `postId` tem mais comentários? Salve o resultado em Parquet (ou CSV se pyarrow não estiver disponível).

In [ ]:
# Sua resposta — Exercício 9

In [ ]:
# ✅ Gabarito — Exercício 9
import requests, pandas as pd, time
from pathlib import Path

def coletar_tudo(url, limit=100):
    pagina, acumulado = 1, []
    while True:
        r = requests.get(url, params={"_page": pagina, "_limit": limit}, timeout=10)
        r.raise_for_status()
        lote = r.json()
        if not lote: break
        acumulado += lote; pagina += 1
        time.sleep(0.2)
    return acumulado

try:
    df9 = pd.DataFrame(coletar_tudo("https://jsonplaceholder.typicode.com/comments"))
    print("Total de comentários:", len(df9))
    print("postId com mais comentários:", df9["postId"].value_counts().idxmax(),
          "(", df9["postId"].value_counts().max(), "comentários )")
    saida = Path("dados_apostila"); saida.mkdir(exist_ok=True)
    try:
        df9.to_parquet(saida / "comments.parquet", index=False); print("Salvo em Parquet")
    except ImportError:
        df9.to_csv(saida / "comments.csv", index=False); print("Salvo em CSV")
except requests.exceptions.RequestException as e:
    print("Erro de rede:", type(e).__name__)

### Exercício 10 — Mini projeto: coletor com cache
Implemente `obter_usuario(id)` que:
1. Verifica se existe `cache/usuario_{id}.json`; se existir, carrega dali (**sem** acessar a rede) e informa "cache";
2. Caso contrário, consulta `https://jsonplaceholder.typicode.com/users/{id}`, salva o JSON em `cache/` e informa "rede";
3. Retorna o dicionário do usuário.

Chame duas vezes para `id=2` e observe o comportamento.

In [ ]:
# Sua resposta — Exercício 10

In [ ]:
# ✅ Gabarito — Exercício 10
import json, requests
from pathlib import Path

CACHE = Path("dados_apostila/cache"); CACHE.mkdir(parents=True, exist_ok=True)

def obter_usuario(id):
    arq = CACHE / f"usuario_{id}.json"
    if arq.exists():
        print(f"[cache] usuário {id}")
        return json.loads(arq.read_text(encoding="utf-8"))
    r = requests.get(f"https://jsonplaceholder.typicode.com/users/{id}", timeout=10)
    r.raise_for_status()
    dados = r.json()
    arq.write_text(json.dumps(dados, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"[rede]  usuário {id}")
    return dados

try:
    print(obter_usuario(2)["name"])
    print(obter_usuario(2)["name"])
except requests.exceptions.RequestException as e:
    print("Erro de rede:", type(e).__name__)

### Exercício 11 (desafio) — XML → análise
Dado o XML de pedidos abaixo, calcule o valor total por cliente (quantidade × preço unitário) e exiba em ordem decrescente. Salve o resultado em `totais_clientes.csv`.

```xml
<pedidos>
  <pedido id="1" cliente="Ana"><item qtd="2" preco="50.0">Caderno</item><item qtd="1" preco="120.0">Mochila</item></pedido>
  <pedido id="2" cliente="Bruno"><item qtd="5" preco="3.5">Caneta</item></pedido>
  <pedido id="3" cliente="Ana"><item qtd="1" preco="80.0">Estojo</item></pedido>
</pedidos>
```

In [ ]:
# Sua resposta — Exercício 11
xml_ex11 = """<pedidos>
  <pedido id="1" cliente="Ana"><item qtd="2" preco="50.0">Caderno</item><item qtd="1" preco="120.0">Mochila</item></pedido>
  <pedido id="2" cliente="Bruno"><item qtd="5" preco="3.5">Caneta</item></pedido>
  <pedido id="3" cliente="Ana"><item qtd="1" preco="80.0">Estojo</item></pedido>
</pedidos>"""

In [ ]:
# ✅ Gabarito — Exercício 11
import xml.etree.ElementTree as ET, pandas as pd
from collections import defaultdict
from pathlib import Path

root11 = ET.fromstring(xml_ex11)
totais = defaultdict(float)
for pedido in root11.findall("pedido"):
    for item in pedido.findall("item"):
        totais[pedido.get("cliente")] += int(item.get("qtd")) * float(item.get("preco"))

df11 = pd.DataFrame(sorted(totais.items(), key=lambda x: -x[1]), columns=["cliente", "total"])
Path("dados_apostila").mkdir(exist_ok=True)
df11.to_csv("dados_apostila/totais_clientes.csv", index=False)
print(df11)

### Exercício 12 (desafio) — Scraping ético com paginação
Raspe as **3 primeiras páginas** de `https://quotes.toscrape.com/`, com pausa de 1 s entre requisições, e produza um DataFrame com colunas `autor`, `texto`, `n_tags`. Responda: qual autor aparece mais vezes? Salve em `quotes.jsonl` (JSON Lines).

In [ ]:
# Sua resposta — Exercício 12

In [ ]:
# ✅ Gabarito — Exercício 12
import requests, time, pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

BASE = "https://quotes.toscrape.com"
HEAD = {"User-Agent": "ApostilaAPIs/1.0 (estudo)"}
linhas = []
try:
    for pag in range(1, 4):
        r = requests.get(f"{BASE}/page/{pag}/", headers=HEAD, timeout=10); r.raise_for_status()
        for q in BeautifulSoup(r.text, "html.parser").select("div.quote"):
            linhas.append({"autor": q.select_one("small.author").text,
                           "texto": q.select_one("span.text").text.strip("“”"),
                           "n_tags": len(q.select("a.tag"))})
        time.sleep(1)
    df12 = pd.DataFrame(linhas)
    print(f"{len(df12)} citações | autor mais frequente: {df12['autor'].value_counts().idxmax()}")
    Path("dados_apostila").mkdir(exist_ok=True)
    df12.to_json("dados_apostila/quotes.jsonl", orient="records", lines=True, force_ascii=False)
    print(df12.head())
except requests.exceptions.RequestException as e:
    print("Erro de rede:", type(e).__name__)

---

# 📚 Referências e leituras complementares

- Documentação `requests`: https://requests.readthedocs.io
- Documentação `BeautifulSoup`: https://www.crummy.com/software/BeautifulSoup/bs4/doc/
- Documentação `pandas` — IO tools: https://pandas.pydata.org/docs/user_guide/io.html
- `pathlib`: https://docs.python.org/3/library/pathlib.html
- MDN — HTTP: https://developer.mozilla.org/pt-BR/docs/Web/HTTP
- Fielding, R. *Architectural Styles and the Design of Network-based Software Architectures* (2000) — origem do REST.
- APIs públicas para praticar: JSONPlaceholder, ViaCEP, IBGE, AwesomeAPI, Open-Meteo, GitHub API, PokéAPI.
- Sites para praticar scraping: quotes.toscrape.com, books.toscrape.com.

---
*Apostila gerada para fins educacionais. Bons estudos! 🚀*